# 🛠️ **Testing Tool Providers Dynamically**

This notebook will:

- List all registered tools from the database, displaying their names and GUIDs explicitly.
- Dynamically load the corresponding tool scripts from the `extensions` directory.
- Instantiate each tool using the shared `ToolProviderBase` infrastructure.
- Test tool invocation explicitly for both success and failure cases.
- Verify the tool invocation logging clearly.


### Configuring Project Path

This cell ensures the notebook can locate the project's modules by adding the project root directory to Python’s import path (`sys.path`). This setup is required for relative imports from the main application (`app`) to work correctly within the notebook environment.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

c:\Repos\codecritic


In [2]:
from sqlalchemy import create_engine
from app.db.base import Base
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[2]
DB_PATH = PROJECT_ROOT / "experiments" / "codecritic.sqlite3"

engine = create_engine(f"sqlite:///{DB_PATH}")

# Explicitly drop and recreate your database schema
Base.metadata.drop_all(bind=engine)
Base.metadata.create_all(bind=engine)

print("✅ Database schema reset and recreated successfully.")

✅ Database schema reset and recreated successfully.


In [3]:
import sqlite3
from app.utilities.db import DB_PATH

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

schema = cur.execute("PRAGMA table_info(tool_invocation_log)").fetchall()
print(schema)

conn.close()


[(0, 'id', 'INTEGER', 1, None, 1), (1, 'experiment_id', 'VARCHAR', 1, None, 0), (2, 'round', 'INTEGER', 1, None, 0), (3, 'tool_provider_name', 'VARCHAR', 1, None, 0), (4, 'tool_provider_guid', 'VARCHAR', 1, None, 0), (5, 'invocation_parameters', 'VARCHAR', 1, None, 0), (6, 'stdout', 'VARCHAR', 0, None, 0), (7, 'stderr', 'VARCHAR', 0, None, 0), (8, 'return_code', 'INTEGER', 1, None, 0), (9, 'success', 'BOOLEAN', 1, None, 0), (10, 'error_message', 'VARCHAR', 0, None, 0), (11, 'timestamp', 'DATETIME', 1, None, 0)]


## 🔧 Database Initialization

In this step, we establish a connection to the SQLite database and explicitly initialize it, creating tables as defined in our SQLAlchemy models.

This step ensures that the database schema matches our SQLAlchemy definitions.

In [4]:
from sqlalchemy import select
from sqlalchemy.orm import Session
from app.db.models import ToolConfig
from app.db.seeders.seed_prompts import seed_prompts
from app.db.seeders.seed_tools import seed_tools

with Session(bind=engine) as session:
    seed_prompts(session)
    seed_tools(session)
    tools = session.execute(select(ToolConfig)).scalars().all()

print("✅ Data seeded successfully.")

print("Registered tools:")
for tool in tools:
    print(f"- {tool.name}: {tool.guid}")
print(PROJECT_ROOT)


Seeded AgentPrompt GUID: f4853420-9dcf-41f3-aa50-24616cf6c08b
Seeded SystemPrompt GUID: 95b8d424-f121-4e38-9cda-a0159792953f
Seeded tool configurations successfully.
✅ Data seeded successfully.
Registered tools:
- Black Formatter: 23c2eb35-4a2a-41f9-98e2-d7bd59bf2f30
- SonarCloud Analyzer: b0e65c71-bed2-431a-984b-474be1c62bd2
- Ruff Linter: 57d0c7c2-b324-44e2-bf49-83c73dd1c096
- Radon Analyzer: 0c6ef746-2614-44dc-8750-c438c9e624c6
- Mypy Type Checker: 31351a80-c5e4-4f06-8442-8840f1f4f754
- Docformatter Formatter: 1a8dac64-9e6d-4ecc-b13a-9fa9cbf9fb32
- Symbol Graph Analyzer: 271c9ed5-4658-4248-961d-3573b7d93327
c:\Repos\codecritic


## 📦 Dynamic Tool Loader

We dynamically load tool scripts explicitly by GUID from the extensions folder, instantiate them using the common `ToolProviderBase` class, and test invocation.


In [5]:
# Minimal Tool Validation Notebook Cell
import sys
import subprocess
import importlib
from pathlib import Path
from sqlalchemy.orm import Session
from sqlalchemy import select, create_engine
from app.abstract_classes.tool_provider_base import ToolProviderBase
from app.db.models import ToolConfig
from app.factories.logging_provider import LoggingProvider

# Setup
PROJECT_ROOT = Path.cwd().parents[2]
DB_PATH = PROJECT_ROOT / "experiments" / "codecritic.sqlite3"
engine = create_engine(f"sqlite:///{DB_PATH}")
logger = LoggingProvider()

# Prepare test files
tests_dir = PROJECT_ROOT / "tests"
tests_dir.mkdir(parents=True, exist_ok=True)
valid_test_file = tests_dir / "example.py"
valid_test_file.write_text("print('hello world')\n", encoding="utf-8")
invalid_test_file = tests_dir / "nonexistent_file.py"

print(f"Valid test file: {valid_test_file}")
print(f"Invalid test file: {invalid_test_file}")

# Load tools
with Session(bind=engine) as session:
    tools = session.execute(select(ToolConfig)).scalars().all()

# Loader helper
def load_tool_provider(cfg, logger):
    spec = importlib.util.spec_from_file_location(cfg.guid, cfg.artifact_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[cfg.guid] = module
    spec.loader.exec_module(module)
    cls = next(
        c for c in vars(module).values()
        if isinstance(c, type) and issubclass(c, ToolProviderBase) and c is not ToolProviderBase
    )
    return cls(cfg, logger=logger)

def describe_result(result):
    if isinstance(result, subprocess.CompletedProcess):
        print("Return code:", result.returncode)
        if result.stdout:
            print("STDOUT:\n", result.stdout)
        if result.stderr:
            print("STDERR:\n", result.stderr)
    elif isinstance(result, str):
        print("Structured output:\n", result)
    else:
        print("Unknown result type:", type(result).__name__)

print("\n🧪 Tool Validation Results")
for tool in tools:
    print(f"\n🛠️ Tool: {tool.name} (GUID: {tool.guid})")
    prov = load_tool_provider(tool, logger)

    # Valid test
    print("\n🔹 Valid invocation (expect success):")
    try:
        result = prov.run(str(valid_test_file), experiment_id="valid_test", round=1)
        describe_result(result)
    except Exception as e:
        print("❌ Unexpected error:", e)

    # Invalid test
    print("\n🔹 Invalid invocation (expect failure):")
    try:
        result = prov.run(str(invalid_test_file), experiment_id="invalid_test", round=1)
        describe_result(result)
        print("❌ Unexpected success")
    except Exception as e:
        print("✅ Caught expected failure:", e)

    print("-" * 60)


Valid test file: c:\Repos\codecritic\tests\example.py
Invalid test file: c:\Repos\codecritic\tests\nonexistent_file.py

🧪 Tool Validation Results

🛠️ Tool: Black Formatter (GUID: 23c2eb35-4a2a-41f9-98e2-d7bd59bf2f30)

🔹 Valid invocation (expect success):
Return code: 0

🔹 Invalid invocation (expect failure):


Usage: python -m black [OPTIONS] SRC ...
Try 'python -m black -h' for help.

Error: Invalid value for 'SRC ...': Path 'c:\\Repos\\codecritic\\tests\\nonexistent_file.py' does not exist.

Critical tool run error logged: black failed: Usage: python -m black [OPTIONS] SRC ...
Try 'python -m black -h' for help.

Error: Invalid value for 'SRC ...': Path 'c:\\Repos\\codecritic\\tests\\nonexistent_file.py' does not exist.

Critical tool run error logged: c:\Repos\codecritic\tests\nonexistent_file.py does not exist
Critical tool run error logged: ruff failed: 


✅ Caught expected failure: black failed: Usage: python -m black [OPTIONS] SRC ...
Try 'python -m black -h' for help.

Error: Invalid value for 'SRC ...': Path 'c:\\Repos\\codecritic\\tests\\nonexistent_file.py' does not exist.

------------------------------------------------------------

🛠️ Tool: SonarCloud Analyzer (GUID: b0e65c71-bed2-431a-984b-474be1c62bd2)

🔹 Valid invocation (expect success):
Return code: 0
STDOUT:
 {}

🔹 Invalid invocation (expect failure):
✅ Caught expected failure: c:\Repos\codecritic\tests\nonexistent_file.py does not exist
------------------------------------------------------------

🛠️ Tool: Ruff Linter (GUID: 57d0c7c2-b324-44e2-bf49-83c73dd1c096)

🔹 Valid invocation (expect success):
Return code: 0
STDOUT:
 All checks passed!


🔹 Invalid invocation (expect failure):
✅ Caught expected failure: ruff failed: 
------------------------------------------------------------

🛠️ Tool: Radon Analyzer (GUID: 0c6ef746-2614-44dc-8750-c438c9e624c6)

🔹 Valid invocation (

Critical tool run error logged: c:\Repos\codecritic\tests\nonexistent_file.py not found


Return code: 0

🔹 Invalid invocation (expect failure):
✅ Caught expected failure: c:\Repos\codecritic\tests\nonexistent_file.py not found
------------------------------------------------------------

🛠️ Tool: Mypy Type Checker (GUID: 31351a80-c5e4-4f06-8442-8840f1f4f754)

🔹 Valid invocation (expect success):
Return code: 0
STDOUT:
 Success: no issues found in 1 source file


🔹 Invalid invocation (expect failure):


mypy: can't read file 'c:\Repos\codecritic\tests\nonexistent_file.py': No such file or directory

Critical tool run error logged: mypy execution error: mypy: can't read file 'c:\Repos\codecritic\tests\nonexistent_file.py': No such file or directory

[Errno 2] No such file or directory: 'c:\\Repos\\codecritic\\tests\\nonexistent_file.py'



✅ Caught expected failure: mypy execution error: mypy: can't read file 'c:\Repos\codecritic\tests\nonexistent_file.py': No such file or directory

------------------------------------------------------------

🛠️ Tool: Docformatter Formatter (GUID: 1a8dac64-9e6d-4ecc-b13a-9fa9cbf9fb32)

🔹 Valid invocation (expect success):
Return code: 0

🔹 Invalid invocation (expect failure):


Critical tool run error logged: docformatter failed: [Errno 2] No such file or directory: 'c:\\Repos\\codecritic\\tests\\nonexistent_file.py'

Critical tool run error logged: [Errno 2] No such file or directory: 'c:\\Repos\\codecritic\\tests\\nonexistent_file.py'


✅ Caught expected failure: docformatter failed: [Errno 2] No such file or directory: 'c:\\Repos\\codecritic\\tests\\nonexistent_file.py'

------------------------------------------------------------

🛠️ Tool: Symbol Graph Analyzer (GUID: 271c9ed5-4658-4248-961d-3573b7d93327)

🔹 Valid invocation (expect success):
Return code: 0
STDOUT:
 {
  "example": {
    "calls": [
      "print"
    ]
  }
}

🔹 Invalid invocation (expect failure):
✅ Caught expected failure: [Errno 2] No such file or directory: 'c:\\Repos\\codecritic\\tests\\nonexistent_file.py'
------------------------------------------------------------


## 🧪 Testing Tool Invocations

We explicitly test both successful and intentional failure cases for each tool to verify correct invocation and logging.


In [6]:
import sys
import importlib
from pathlib import Path
from sqlalchemy.orm import Session
from sqlalchemy import select, create_engine
from app.abstract_classes.tool_provider_base import ToolProviderBase
from app.db.models import ToolConfig
from app.factories.logging_provider import LoggingProvider

PROJECT_ROOT = Path.cwd().parents[2]
DB_PATH = PROJECT_ROOT / "experiments" / "codecritic.sqlite3"
engine = create_engine(f"sqlite:///{DB_PATH}")
logger = LoggingProvider()

# Create explicitly valid and invalid test files
valid_test_file = PROJECT_ROOT / "tests" / "example.py"
valid_test_file.parent.mkdir(parents=True, exist_ok=True)
valid_test_file.write_text("print('hello world')\n", encoding="utf-8")

invalid_test_file = PROJECT_ROOT / "tests" / "nonexistent_file.py"  # intentionally nonexistent

with Session(bind=engine) as session:
    tools = session.execute(select(ToolConfig)).scalars().all()

def load_tool_provider(tool_config: ToolConfig, logger: LoggingProvider) -> ToolProviderBase:
    tool_script_path = Path(tool_config.artifact_path)
    spec = importlib.util.spec_from_file_location(tool_config.guid, tool_script_path)
    tool_module = importlib.util.module_from_spec(spec)
    sys.modules[tool_config.guid] = tool_module
    spec.loader.exec_module(tool_module)

    tool_class = next(
        cls for cls in vars(tool_module).values()
        if isinstance(cls, type)
        and issubclass(cls, ToolProviderBase)
        and cls is not ToolProviderBase
    )

    return tool_class(tool_config, logger=logger)

print("🧪 Final Corrected Tool Invocations\n")

for tool_config in tools:
    print(f"\n🛠️ Tool: {tool_config.name} (GUID: {tool_config.guid})")

    provider = load_tool_provider(tool_config, logger)

    # Explicit VALID TEST
    try:
        print("✅ Valid invocation test:")
        result = provider.run(str(valid_test_file), experiment_id="valid_test", round=1)
        if isinstance(result, str):
            print("Structured output:", result)
        else:
            print("Return code:", result.returncode)
            print("STDOUT:", result.stdout.strip())
            print("STDERR:", result.stderr.strip())
    except Exception as e:
        print("❌ Unexpected error during valid case:", e)

    # Explicit INVALID TEST (expected to fail)
    try:
        print("❌ Invalid invocation test (expected failure):")
        result = provider.run(str(invalid_test_file), experiment_id="invalid_test", round=1)
        if isinstance(result, str):
            print("Structured output:", result)
        else:
            print("Return code:", result.returncode)
            print("STDOUT:", result.stdout.strip())
            print("STDERR:", result.stderr.strip())
    except Exception as e:
        print("✅ Caught expected failure:", e)


🧪 Final Corrected Tool Invocations


🛠️ Tool: Black Formatter (GUID: 23c2eb35-4a2a-41f9-98e2-d7bd59bf2f30)
✅ Valid invocation test:


Usage: python -m black [OPTIONS] SRC ...
Try 'python -m black -h' for help.

Error: Invalid value for 'SRC ...': Path 'c:\\Repos\\codecritic\\tests\\nonexistent_file.py' does not exist.



Return code: 0
STDOUT: 
STDERR: 
❌ Invalid invocation test (expected failure):


Critical tool run error logged: black failed: Usage: python -m black [OPTIONS] SRC ...
Try 'python -m black -h' for help.

Error: Invalid value for 'SRC ...': Path 'c:\\Repos\\codecritic\\tests\\nonexistent_file.py' does not exist.

Critical tool run error logged: c:\Repos\codecritic\tests\nonexistent_file.py does not exist


✅ Caught expected failure: black failed: Usage: python -m black [OPTIONS] SRC ...
Try 'python -m black -h' for help.

Error: Invalid value for 'SRC ...': Path 'c:\\Repos\\codecritic\\tests\\nonexistent_file.py' does not exist.


🛠️ Tool: SonarCloud Analyzer (GUID: b0e65c71-bed2-431a-984b-474be1c62bd2)
✅ Valid invocation test:
Return code: 0
STDOUT: {}
STDERR: 
❌ Invalid invocation test (expected failure):
✅ Caught expected failure: c:\Repos\codecritic\tests\nonexistent_file.py does not exist

🛠️ Tool: Ruff Linter (GUID: 57d0c7c2-b324-44e2-bf49-83c73dd1c096)
✅ Valid invocation test:
Return code: 0
STDOUT: All checks passed!
STDERR: 
❌ Invalid invocation test (expected failure):


...

Return code: 0
STDOUT: 
STDERR: 
❌ Invalid invocation test (expected failure):
✅ Caught expected failure: c:\Repos\codecritic\tests\nonexistent_file.py not found

🛠️ Tool: Mypy Type Checker (GUID: 31351a80-c5e4-4f06-8442-8840f1f4f754)
✅ Valid invocation test:


mypy: can't read file 'c:\Repos\codecritic\tests\nonexistent_file.py': No such file or directory

Critical tool run error logged: mypy execution error: mypy: can't read file 'c:\Repos\codecritic\tests\nonexistent_file.py': No such file or directory



Return code: 0
STDOUT: Success: no issues found in 1 source file
STDERR: 
❌ Invalid invocation test (expected failure):
✅ Caught expected failure: mypy execution error: mypy: can't read file 'c:\Repos\codecritic\tests\nonexistent_file.py': No such file or directory


🛠️ Tool: Docformatter Formatter (GUID: 1a8dac64-9e6d-4ecc-b13a-9fa9cbf9fb32)
✅ Valid invocation test:


[Errno 2] No such file or directory: 'c:\\Repos\\codecritic\\tests\\nonexistent_file.py'

Critical tool run error logged: docformatter failed: [Errno 2] No such file or directory: 'c:\\Repos\\codecritic\\tests\\nonexistent_file.py'

Critical tool run error logged: [Errno 2] No such file or directory: 'c:\\Repos\\codecritic\\tests\\nonexistent_file.py'


Return code: 0
STDOUT: 
STDERR: 
❌ Invalid invocation test (expected failure):
✅ Caught expected failure: docformatter failed: [Errno 2] No such file or directory: 'c:\\Repos\\codecritic\\tests\\nonexistent_file.py'


🛠️ Tool: Symbol Graph Analyzer (GUID: 271c9ed5-4658-4248-961d-3573b7d93327)
✅ Valid invocation test:
Return code: 0
STDOUT: {
  "example": {
    "calls": [
      "print"
    ]
  }
}
STDERR: 
❌ Invalid invocation test (expected failure):
✅ Caught expected failure: [Errno 2] No such file or directory: 'c:\\Repos\\codecritic\\tests\\nonexistent_file.py'


## 📜 Verifying Logs Explicitly

Explicitly query the logs to confirm the tool invocation records were correctly created.


In [7]:
import sqlite3
import json

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

print("✅ Tool Invocation Logs:")
for row in cur.execute("SELECT experiment_id, round, tool_provider_name, tool_provider_guid, success, error_message FROM tool_invocation_log"):
    print(row)

conn.close()


✅ Tool Invocation Logs:
('valid_test', 1, 'BlackToolProvider', '23c2eb35-4a2a-41f9-98e2-d7bd59bf2f30', 1, None)
('invalid_test', 1, 'BlackToolProvider', '23c2eb35-4a2a-41f9-98e2-d7bd59bf2f30', 0, "black failed: Usage: python -m black [OPTIONS] SRC ...\nTry 'python -m black -h' for help.\n\nError: Invalid value for 'SRC ...': Path 'c:\\\\Repos\\\\codecritic\\\\tests\\\\nonexistent_file.py' does not exist.\n")
('valid_test', 1, 'SonarCloudToolProvider', 'b0e65c71-bed2-431a-984b-474be1c62bd2', 1, None)
('invalid_test', 1, 'SonarCloudToolProvider', 'b0e65c71-bed2-431a-984b-474be1c62bd2', 0, 'c:\\Repos\\codecritic\\tests\\nonexistent_file.py does not exist')
('valid_test', 1, 'RuffToolProvider', '57d0c7c2-b324-44e2-bf49-83c73dd1c096', 1, None)
('invalid_test', 1, 'RuffToolProvider', '57d0c7c2-b324-44e2-bf49-83c73dd1c096', 0, 'ruff failed: ')
('valid_test', 1, 'RadonToolProvider', '0c6ef746-2614-44dc-8750-c438c9e624c6', 1, None)
('invalid_test', 1, 'RadonToolProvider', '0c6ef746-2614-44dc-87